In [1]:
from __imports__ import *

In [2]:
# Just to test the connection
dbo.query("""
SELECT TOP 1 *
FROM infra_private_punctuality
""")

TRAIN_ID,TRAIN_NO,TRAIN_SERV_ID,ORD_NO,PTCAR_NO,PLANNED_DATETIME_ARR,REAL_DATETIME_ARR,PLANNED_DATETIME_DEP,REAL_DATETIME_DEP,DELAY_ARR,DELAY_DEP,LINE_DIR_DEP_ID,LINE_DIR_ARR_ID,PTCAR_FROM_NO,PTCAR_TO_NO,TRAIN_CANCELED,PTCAR_CANCELED,DETOUR_ARR
str,str,i64,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,i64,i64,str,str,str,str,str
"""119953363""","""3031""",7,"""135""","""1282""",2025-01-14 10:55:00,2025-01-14 10:54:27,2025-01-14 10:55:00,2025-01-14 10:54:27,"""-32""","""-32""",129,129,"""1278""","""64""","""0""","""0""","""0"""


## This query prepares the dataset for the analysis

It does the following:

Selects data from January 2025


Keeps only trains that pass through both Brussels-North (221) and Brussels-Midi (220)

Keeps only operational points located between North and Midi, regardless of train direction

Converts arrival delay from seconds to minutes

Sorts observations chronologically

In [ ]:
junction_8_9 = dbo.query("""
WITH train_limits AS (

    SELECT
        TRAIN_ID,

        MIN(
            CASE
                WHEN PTCAR_NO = '221'
                THEN TRY_CAST(ORD_NO AS INT)
            END
        ) AS ORD_NORD,

        MIN(
            CASE
                WHEN PTCAR_NO = '220'
                THEN TRY_CAST(ORD_NO AS INT)
            END
        ) AS ORD_MIDI

    FROM infra_private_punctuality

    WHERE PLANNED_DATETIME_ARR >= '2025-01-01'
      AND PLANNED_DATETIME_ARR <  '2025-02-01'

    GROUP BY TRAIN_ID

    HAVING
        MIN(
            CASE
                WHEN PTCAR_NO = '221'
                THEN TRY_CAST(ORD_NO AS INT)
            END
        ) IS NOT NULL

        AND

        MIN(
            CASE
                WHEN PTCAR_NO = '220'
                THEN TRY_CAST(ORD_NO AS INT)
            END
        ) IS NOT NULL
)

SELECT
    CAST(p.PLANNED_DATETIME_ARR AS DATE) AS DAY,

    p.TRAIN_ID,
    p.TRAIN_NO,
    p.ORD_NO,
    p.PTCAR_NO,

    op.Complete_name_in_French AS PTCAR_NAME,

    p.PLANNED_DATETIME_ARR,
    p.REAL_DATETIME_ARR,

    TRY_CAST(p.DELAY_ARR AS FLOAT) / 60.0 AS DELAY_ARR_MIN

FROM infra_private_punctuality p

INNER JOIN train_limits t
    ON p.TRAIN_ID = t.TRAIN_ID

LEFT JOIN infra_operational_points op
    ON p.PTCAR_NO = op.PTCAR_ID

WHERE
    p.PLANNED_DATETIME_ARR >= '2025-01-01'
    AND p.PLANNED_DATETIME_ARR < '2025-02-01'


    -- Only points located between North and Midi
    AND TRY_CAST(p.ORD_NO AS INT)
        BETWEEN
            CASE
                WHEN t.ORD_NORD < t.ORD_MIDI
                THEN t.ORD_NORD
                ELSE t.ORD_MIDI
            END
        AND
            CASE
                WHEN t.ORD_NORD > t.ORD_MIDI
                THEN t.ORD_NORD
                ELSE t.ORD_MIDI
            END

ORDER BY
    p.PLANNED_DATETIME_ARR,
    p.TRAIN_ID
""")

In [4]:
import polars as pl

## Just to check the size and time coverage of the extracted 8–9 AM dataset before starting the delay propagation analysis

In [5]:
junction_8_9.select([
    pl.len().alias("N_ROWS"),
    pl.col("TRAIN_ID").n_unique().alias("N_TRAINS"),
    pl.col("DAY").n_unique().alias("N_DAYS"),
    pl.col("PLANNED_DATETIME_ARR").min().alias("MIN_TIME"),
    pl.col("PLANNED_DATETIME_ARR").max().alias("MAX_TIME")
])

N_ROWS,N_TRAINS,N_DAYS,MIN_TIME,MAX_TIME
u32,u32,u32,datetime[μs],datetime[μs]
10105,2034,31,2025-01-01 08:00:00,2025-01-31 08:59:00


## Manually check that the dataset was extracted correctly before continuing the analysis

In [6]:
junction_8_9.head(50)

DAY,TRAIN_ID,TRAIN_NO,ORD_NO,PTCAR_NO,PTCAR_NAME,PLANNED_DATETIME_ARR,REAL_DATETIME_ARR,DELAY_ARR_MIN
date,str,str,str,str,str,datetime[μs],datetime[μs],f64
2025-01-01,"""119534432""","""529""","""150""","""220""","""Bruxelles-Midi""",2025-01-01 08:00:00,2025-01-01 08:05:06,5.1
2025-01-01,"""119539766""","""506""","""136""","""215""","""Bruxelles-Central""",2025-01-01 08:00:00,2025-01-01 08:00:00,0.0
2025-01-01,"""119547303""","""1906""","""127""","""221""","""Bruxelles-Nord""",2025-01-01 08:00:00,2025-01-01 08:00:29,0.483333
2025-01-01,"""119547770""","""3778""","""126""","""220""","""Bruxelles-Midi""",2025-01-01 08:00:00,2025-01-01 07:59:20,-0.65
2025-01-01,"""119521617""","""3128""","""112""","""217""","""Bruxelles-Chapelle""",2025-01-01 08:01:00,2025-01-01 08:15:52,14.866667
…,…,…,…,…,…,…,…,…
2025-01-01,"""119522434""","""2806""","""140""","""217""","""Bruxelles-Chapelle""",2025-01-01 08:15:00,2025-01-01 08:16:06,1.1
2025-01-01,"""119522434""","""2806""","""139""","""305""","""Bruxelles-Midi-Gril JNM""",2025-01-01 08:15:00,2025-01-01 08:14:38,-0.366667
2025-01-01,"""119548834""","""2229""","""110""","""215""","""Bruxelles-Central""",2025-01-01 08:15:00,2025-01-01 08:14:51,-0.133333


## Create a train-level summary that makes it easier to compare trains and identify potential leader–follower relationships

It does the following:

Groups the data by day, train ID, and train number

Counts how many operational points were observed for each train with N_POINTS

Finds the first planned arrival time with FIRST_TIME

Finds the last planned arrival time with LAST_TIME

Finds the minimum delay observed for that train with MIN_DELAY

Finds the maximum delay observed for that train with MAX_DELAY

Sorts the trains by day and then by their first planned time

In [7]:
trains_8_9 = (
    junction_8_9
    .group_by([
        "DAY",
        "TRAIN_ID",
        "TRAIN_NO"
    ])
    .agg([
        pl.len().alias("N_POINTS"),
        pl.col("PLANNED_DATETIME_ARR").min().alias("FIRST_TIME"),
        pl.col("PLANNED_DATETIME_ARR").max().alias("LAST_TIME"),
        pl.col("DELAY_ARR_MIN").min().alias("MIN_DELAY"),
        pl.col("DELAY_ARR_MIN").max().alias("MAX_DELAY")
    ])
    .sort([
        "DAY",
        "FIRST_TIME"
    ])
)

trains_8_9.head(50)

DAY,TRAIN_ID,TRAIN_NO,N_POINTS,FIRST_TIME,LAST_TIME,MIN_DELAY,MAX_DELAY
date,str,str,u32,datetime[μs],datetime[μs],f64,f64
2025-01-01,"""119539766""","""506""",3,2025-01-01 08:00:00,2025-01-01 08:05:00,0.0,1.666667
2025-01-01,"""119547303""","""1906""",1,2025-01-01 08:00:00,2025-01-01 08:00:00,0.483333,0.483333
2025-01-01,"""119534432""","""529""",1,2025-01-01 08:00:00,2025-01-01 08:00:00,5.1,5.1
2025-01-01,"""119547770""","""3778""",1,2025-01-01 08:00:00,2025-01-01 08:00:00,-0.65,-0.65
2025-01-01,"""119550497""","""3107""",5,2025-01-01 08:01:00,2025-01-01 08:08:00,-0.95,0.183333
…,…,…,…,…,…,…,…
2025-01-02,"""119557404""","""1906""",1,2025-01-02 08:00:00,2025-01-02 08:00:00,2.3,2.3
2025-01-02,"""119559392""","""3107""",3,2025-01-02 08:00:00,2025-01-02 08:05:00,-0.15,0.05
2025-01-02,"""119564377""","""3778""",1,2025-01-02 08:00:00,2025-01-02 08:00:00,0.0,0.0


## Build leader–follower train pairs at the same operational point, so their delays and time separation can be analyzed later

It does the following:

Removes rows with missing train, operational point, planned time, or delay information

Sorts observations by day, operational point, and planned arrival time

Identifies the previous train at the same operational point as the leader

Stores the leader’s train ID, train number, planned time, and delay

Renames the current train as the follower

Removes observations where no leader exists

Computes the scheduled time difference between the leader and follower as TIME_GAP_MIN.

In [8]:
leader_follower_8_9 = (
    junction_8_9

    .filter(
        pl.col("TRAIN_ID").is_not_null()
        & pl.col("PTCAR_NO").is_not_null()
        & pl.col("PLANNED_DATETIME_ARR").is_not_null()
        & pl.col("DELAY_ARR_MIN").is_not_null()
    )

    .sort([
        "DAY",
        "PTCAR_NO",
        "PLANNED_DATETIME_ARR"
    ])

    .with_columns([
        # Previous train at the same operational point
        pl.col("TRAIN_ID")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("LEADER_TRAIN_ID"),

        pl.col("TRAIN_NO")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("LEADER_TRAIN_NO"),

        pl.col("PLANNED_DATETIME_ARR")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("LEADER_TIME"),

        pl.col("DELAY_ARR_MIN")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("LEADER_DELAY_MIN")
    ])

    .rename({
        "TRAIN_ID": "FOLLOWER_TRAIN_ID",
        "TRAIN_NO": "FOLLOWER_TRAIN_NO",
        "PLANNED_DATETIME_ARR": "FOLLOWER_TIME",
        "DELAY_ARR_MIN": "FOLLOWER_DELAY_MIN"
    })

    .filter(
        pl.col("LEADER_TRAIN_ID").is_not_null()
    )

    .with_columns(
        (
            (pl.col("FOLLOWER_TIME") - pl.col("LEADER_TIME"))
            .dt.total_seconds()
            / 60
        ).alias("TIME_GAP_MIN")
    )
)

## Manually check the results

In [9]:
leader_follower_8_9.select([
    "DAY",
    "PTCAR_NO",
    "PTCAR_NAME",
    "LEADER_TRAIN_NO",
    "FOLLOWER_TRAIN_NO",
    "LEADER_TIME",
    "FOLLOWER_TIME",
    "TIME_GAP_MIN",
    "LEADER_DELAY_MIN",
    "FOLLOWER_DELAY_MIN"
]).head(50)

DAY,PTCAR_NO,PTCAR_NAME,LEADER_TRAIN_NO,FOLLOWER_TRAIN_NO,LEADER_TIME,FOLLOWER_TIME,TIME_GAP_MIN,LEADER_DELAY_MIN,FOLLOWER_DELAY_MIN
date,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64
2025-01-01,"""215""","""Bruxelles-Central""","""506""","""3107""",2025-01-01 08:00:00,2025-01-01 08:03:00,3.0,0.0,-0.433333
2025-01-01,"""215""","""Bruxelles-Central""","""3107""","""3757""",2025-01-01 08:03:00,2025-01-01 08:03:00,0.0,-0.433333,1.316667
2025-01-01,"""215""","""Bruxelles-Central""","""3757""","""1929""",2025-01-01 08:03:00,2025-01-01 08:05:00,2.0,1.316667,-0.4
2025-01-01,"""215""","""Bruxelles-Central""","""1929""","""1978""",2025-01-01 08:05:00,2025-01-01 08:06:00,1.0,-0.4,0.133333
2025-01-01,"""215""","""Bruxelles-Central""","""1978""","""6579""",2025-01-01 08:06:00,2025-01-01 08:08:00,2.0,0.133333,0.85
…,…,…,…,…,…,…,…,…,…
2025-01-01,"""216""","""Bruxelles-Congrès""","""2305""","""3228""",2025-01-01 08:30:00,2025-01-01 08:36:00,6.0,-0.45,0.166667
2025-01-01,"""216""","""Bruxelles-Congrès""","""3228""","""2830""",2025-01-01 08:36:00,2025-01-01 08:40:00,4.0,0.166667,0.4
2025-01-01,"""216""","""Bruxelles-Congrès""","""2830""","""3407""",2025-01-01 08:40:00,2025-01-01 08:40:00,0.0,0.4,-0.333333


## Identify the most recurrent and stable leader–follower train pairs for further delay propagation analysis

It does the following:

Groups the data by each leader train number + follower train number pair

Counts how many times each pair appears: N_OBSERVATIONS

Counts on how many different days the pair appears: N_DAYS

Counts at how many operational points the pair is observed: N_PTCAR

Computes the typical time separation between the two trains using the median: MEDIAN_GAP_MIN

Sorts the pairs so that the most recurrent ones appear first

In [10]:
pair_summary_8_9 = (
    leader_follower_8_9

    .group_by([
        "LEADER_TRAIN_NO",
        "FOLLOWER_TRAIN_NO"
    ])

    .agg([
        # Number of observations of this pair
        pl.len().alias("N_OBSERVATIONS"),

        # Number of different days
        pl.col("DAY")
        .n_unique()
        .alias("N_DAYS"),

        # Number of operational points where the pair appears
        pl.col("PTCAR_NO")
        .n_unique()
        .alias("N_PTCAR"),

        # Typical time gap
        pl.col("TIME_GAP_MIN")
        .median()
        .alias("MEDIAN_GAP_MIN")
    ])

    .sort(
        ["N_DAYS", "N_OBSERVATIONS"],
        descending=True
    )
)

pair_summary_8_9.head(30)

LEADER_TRAIN_NO,FOLLOWER_TRAIN_NO,N_OBSERVATIONS,N_DAYS,N_PTCAR,MEDIAN_GAP_MIN
str,str,u32,u32,u32,f64
"""3407""","""1507""",98,30,6,2.0
"""3707""","""1529""",29,29,1,1.0
"""2331""","""3407""",51,27,3,2.0
"""2830""","""2257""",33,25,2,1.0
"""3429""","""2128""",67,23,6,1.0
…,…,…,…,…,…
"""3979""","""3230""",47,20,5,1.0
"""3657""","""407""",46,20,6,1.0
"""1929""","""2029""",43,20,5,1.0


## the most frequently observed pair

In [11]:
selected_pair = pair_summary_8_9.row(0, named=True)

leader_no = selected_pair["LEADER_TRAIN_NO"]
follower_no = selected_pair["FOLLOWER_TRAIN_NO"]

print("Leader:", leader_no)
print("Follower:", follower_no)

Leader: 3407
Follower: 1507


## Select the chosen Leader–Follower train pair and retrieve its observations

In [12]:
pair_data = (
    leader_follower_8_9
    .filter(
        (pl.col("LEADER_TRAIN_NO") == leader_no)
        & (pl.col("FOLLOWER_TRAIN_NO") == follower_no)
    )
    .sort(["DAY", "FOLLOWER_TIME"])
)

pair_data.select([
    "DAY",
    "PTCAR_NO",

    "LEADER_TRAIN_NO",
    "FOLLOWER_TRAIN_NO",

    "LEADER_TIME",
    "FOLLOWER_TIME",

    "TIME_GAP_MIN",

    "LEADER_DELAY_MIN",
    "FOLLOWER_DELAY_MIN"
]).head(50)

DAY,PTCAR_NO,LEADER_TRAIN_NO,FOLLOWER_TRAIN_NO,LEADER_TIME,FOLLOWER_TIME,TIME_GAP_MIN,LEADER_DELAY_MIN,FOLLOWER_DELAY_MIN
date,str,str,str,datetime[μs],datetime[μs],f64,f64,f64
2025-01-01,"""220""","""3407""","""1507""",2025-01-01 08:30:00,2025-01-01 08:31:00,1.0,-0.5,-1.45
2025-01-01,"""217""","""3407""","""1507""",2025-01-01 08:35:00,2025-01-01 08:38:00,3.0,0.266667,0.733333
2025-01-01,"""305""","""3407""","""1507""",2025-01-01 08:35:00,2025-01-01 08:38:00,3.0,-0.95,-0.516667
2025-01-01,"""216""","""3407""","""1507""",2025-01-01 08:40:00,2025-01-01 08:43:00,3.0,-0.333333,0.116667
2025-01-02,"""305""","""3407""","""1507""",2025-01-02 08:36:00,2025-01-02 08:38:00,2.0,-0.75,-0.933333
…,…,…,…,…,…,…,…,…
2025-01-16,"""216""","""3407""","""1507""",2025-01-16 08:41:00,2025-01-16 08:42:00,1.0,-0.483333,7.733333
2025-01-17,"""305""","""3407""","""1507""",2025-01-17 08:36:00,2025-01-17 08:38:00,2.0,10.033333,-0.3
2025-01-17,"""215""","""3407""","""1507""",2025-01-17 08:38:00,2025-01-17 08:40:00,2.0,11.35,1.733333


## Summarize the number of observations and unique operational points for the selected Leader–Follower pair on each day

In [14]:
pair_data.group_by("DAY").agg([
    pl.len().alias("N_OBSERVATIONS"),
    pl.col("PTCAR_NO").n_unique().alias("N_PTCAR")
]).sort("DAY")

DAY,N_OBSERVATIONS,N_PTCAR
date,u32,u32
2025-01-01,4,4
2025-01-02,2,2
2025-01-03,3,3
2025-01-04,3,3
2025-01-05,3,3
…,…,…
2025-01-27,3,3
2025-01-28,3,3
2025-01-29,4,4


## Prepare the selected Leader–Follower delay observations for Granger analysis by removing missing delays and ordering the data chronologically

In [15]:
granger_data = (
    pair_data
    .select([
        "DAY",
        "PTCAR_NO",
        "LEADER_TIME",
        "FOLLOWER_TIME",
        "LEADER_DELAY_MIN",
        "FOLLOWER_DELAY_MIN"
    ])
    .drop_nulls([
        "LEADER_DELAY_MIN",
        "FOLLOWER_DELAY_MIN"
    ])
    .sort([
        "DAY",
        "FOLLOWER_TIME"
    ])
)

granger_data.head(30)

DAY,PTCAR_NO,LEADER_TIME,FOLLOWER_TIME,LEADER_DELAY_MIN,FOLLOWER_DELAY_MIN
date,str,datetime[μs],datetime[μs],f64,f64
2025-01-01,"""220""",2025-01-01 08:30:00,2025-01-01 08:31:00,-0.5,-1.45
2025-01-01,"""217""",2025-01-01 08:35:00,2025-01-01 08:38:00,0.266667,0.733333
2025-01-01,"""305""",2025-01-01 08:35:00,2025-01-01 08:38:00,-0.95,-0.516667
2025-01-01,"""216""",2025-01-01 08:40:00,2025-01-01 08:43:00,-0.333333,0.116667
2025-01-02,"""305""",2025-01-02 08:36:00,2025-01-02 08:38:00,-0.75,-0.933333
…,…,…,…,…,…
2025-01-09,"""216""",2025-01-09 08:41:00,2025-01-09 08:42:00,1.333333,3.55
2025-01-10,"""220""",2025-01-10 08:32:00,2025-01-10 08:32:00,8.183333,12.616667
2025-01-10,"""305""",2025-01-10 08:36:00,2025-01-10 08:38:00,7.916667,11.083333


In [16]:
# Check the number of observations available for the Granger causality test
print("Number of observations:", granger_data.height)

Number of observations: 98


## Visualize the delay evolution of the selected Leader and Follower pair using the chronologically ordered observations

In [18]:
import plotly.express as px

plot_data = (
    granger_data
    .with_row_index("OBSERVATION")
    .to_pandas()
)

fig = px.line(
    plot_data,
    x="OBSERVATION",
    y=[
        "LEADER_DELAY_MIN",
        "FOLLOWER_DELAY_MIN"
    ],
    title=f"Leader {leader_no} vs Follower {follower_no} : 8-9 AM"
)

fig.update_layout(
    xaxis_title="Ordered observation",
    yaxis_title="Arrival delay (minutes)"
)

fig.show(renderer="browser")

# Granger causality test

## Count the observations available for the selected Leader–Follower pair, on each day and rank the days from most to least observations.

In [19]:
day_counts = (
    pair_data
    .group_by("DAY")
    .agg(
        pl.len().alias("N_OBSERVATIONS")
    )
    .sort(
        "N_OBSERVATIONS",
        descending=True
    )
)

day_counts

DAY,N_OBSERVATIONS
date,u32
2025-01-12,5
2025-01-22,5
2025-01-31,5
2025-01-01,4
2025-01-10,4
…,…
2025-01-02,2
2025-01-06,2
2025-01-11,2


## Select the day with the most observations

In [20]:
selected_day = day_counts.row(0, named=True)["DAY"]
print("Selected day:", selected_day)

Selected day: 2025-01-12


In [21]:
granger_day = (
    pair_data
    .filter(
        pl.col("DAY") == selected_day
    )
    .select([
        "FOLLOWER_TIME",
        "PTCAR_NO",
        "LEADER_DELAY_MIN",
        "FOLLOWER_DELAY_MIN"
    ])
    .drop_nulls([
        "LEADER_DELAY_MIN",
        "FOLLOWER_DELAY_MIN"
    ])
    .sort("FOLLOWER_TIME")
)

granger_day

FOLLOWER_TIME,PTCAR_NO,LEADER_DELAY_MIN,FOLLOWER_DELAY_MIN
datetime[μs],str,f64,f64
2025-01-12 08:31:00,"""220""",0.516667,0.633333
2025-01-12 08:38:00,"""305""",-0.883333,-0.2
2025-01-12 08:39:00,"""217""",0.583333,-0.116667
2025-01-12 08:43:00,"""216""",0.3,2.133333
2025-01-12 08:45:00,"""221""",0.516667,2.416667


In [29]:
print("Number of observations:", granger_day.height)

Number of observations: 5


In [30]:
from statsmodels.tsa.stattools import grangercausalitytests

In [31]:
granger_pd = granger_day.select([
    "FOLLOWER_DELAY_MIN",
    "LEADER_DELAY_MIN"
]).to_pandas()

In [28]:
results_L_to_F = grangercausalitytests(
    granger_pd[
        ["FOLLOWER_DELAY_MIN", "LEADER_DELAY_MIN"]
    ],
    maxlag=2,
    verbose=False
)

/Users/houdamalki/Library/Caches/pypoetry/virtualenvs/cortex-F_jpvhBx-py3.12/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


ValueError: Insufficient observations. Maximum allowable lag is 0

# Granger causality cannot be reliably tested with only 5 observations

## Identify the operational point with the most observations and the best day coverage for the selected Leader–Follower pair between 8 and 9 AM

In [32]:
ptcar_counts = (
    pair_data
    .group_by("PTCAR_NO")
    .agg([
        pl.len().alias("N_OBSERVATIONS"),
        pl.col("DAY").n_unique().alias("N_DAYS")
    ])
    .sort("N_DAYS", descending=True)
)

ptcar_counts

PTCAR_NO,N_OBSERVATIONS,N_DAYS
str,u32,u32
"""305""",29,29
"""216""",26,26
"""220""",17,17
"""217""",14,14
"""215""",8,8
"""221""",4,4


## Select the operational point with the highest day coverage for the chosen Leader–Follower pair

In [33]:
best_ptcar = ptcar_counts.row(0, named=True)["PTCAR_NO"]

print("Selected PTCAR:", best_ptcar)

Selected PTCAR: 305


## Build the monthly Granger dataset for the selected Leader–Follower pair at the operational point with the best day coverage between 8 and 9 AM

In [34]:
granger_month = (
    pair_data
    .filter(
        pl.col("PTCAR_NO") == best_ptcar
    )
    .select([
        "DAY",
        "LEADER_TRAIN_NO",
        "FOLLOWER_TRAIN_NO",
        "LEADER_DELAY_MIN",
        "FOLLOWER_DELAY_MIN",
        "TIME_GAP_MIN"
    ])
    .drop_nulls([
        "LEADER_DELAY_MIN",
        "FOLLOWER_DELAY_MIN"
    ])
    .sort("DAY")
)

granger_month

DAY,LEADER_TRAIN_NO,FOLLOWER_TRAIN_NO,LEADER_DELAY_MIN,FOLLOWER_DELAY_MIN,TIME_GAP_MIN
date,str,str,f64,f64,f64
2025-01-01,"""3407""","""1507""",-0.95,-0.516667,3.0
2025-01-02,"""3407""","""1507""",-0.75,-0.933333,2.0
2025-01-03,"""3407""","""1507""",4.033333,-1.016667,2.0
2025-01-04,"""3407""","""1507""",3.0,-1.066667,3.0
2025-01-05,"""3407""","""1507""",-0.2,6.983333,3.0
…,…,…,…,…,…
2025-01-27,"""3407""","""1507""",4.166667,0.283333,2.0
2025-01-28,"""3407""","""1507""",4.116667,0.233333,2.0
2025-01-29,"""3407""","""1507""",3.1,1.316667,2.0


In [35]:
print("Number of observations:", granger_month.height)
print("Number of days:", granger_month["DAY"].n_unique())

Number of observations: 29
Number of days: 29


In [36]:
granger_month.select([
    "DAY",
    "LEADER_DELAY_MIN",
    "FOLLOWER_DELAY_MIN"
])

DAY,LEADER_DELAY_MIN,FOLLOWER_DELAY_MIN
date,f64,f64
2025-01-01,-0.95,-0.516667
2025-01-02,-0.75,-0.933333
2025-01-03,4.033333,-1.016667
2025-01-04,3.0,-1.066667
2025-01-05,-0.2,6.983333
…,…,…
2025-01-27,4.166667,0.283333
2025-01-28,4.116667,0.233333
2025-01-29,3.1,1.316667


## Prepare the two delay series in the format required by the Granger causality test

In [39]:
granger_pd = (
    granger_month
    .select([
        "FOLLOWER_DELAY_MIN",
        "LEADER_DELAY_MIN"
    ])
    .to_pandas()
)

## Test whether past Leader delays help predict Follower delays over the monthly series, using lags of 1 and 2 observations.

In [38]:
from statsmodels.tsa.stattools import grangercausalitytests

results_L_to_F = grangercausalitytests(
    granger_pd[
        ["FOLLOWER_DELAY_MIN", "LEADER_DELAY_MIN"]
    ],
    maxlag=2,
    verbose=False
)

for lag in range(1, 3):
    p_value = results_L_to_F[lag][0]["ssr_ftest"][1]
    print(f"Lag {lag}: p-value = {p_value:.4f}")

Lag 1: p-value = 0.3498
Lag 2: p-value = 0.4856


/Users/houdamalki/Library/Caches/pypoetry/virtualenvs/cortex-F_jpvhBx-py3.12/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


## Test whether past Follower delays help predict Leader delays

In [40]:
results_F_to_L = grangercausalitytests(
    granger_pd[
        ["LEADER_DELAY_MIN", "FOLLOWER_DELAY_MIN"]
    ],
    maxlag=2,
    verbose=False
)

for lag in range(1, 3):
    p_value = results_F_to_L[lag][0]["ssr_ftest"][1]
    print(f"Lag {lag}: p-value = {p_value:.4f}")

Lag 1: p-value = 0.0860
Lag 2: p-value = 0.2914


/Users/houdamalki/Library/Caches/pypoetry/virtualenvs/cortex-F_jpvhBx-py3.12/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


# With a 5% significance level, none of the p-values are statistically significant

# The Granger causality test does not detect a significant temporal predictive relationship in either direction

In [ ]:
intra_day = (
    junction_8_9
    .filter(
        pl.col("PLANNED_DATETIME_ARR").is_not_null()
        & pl.col("DELAY_ARR_MIN").is_not_null()
        & pl.col("PTCAR_NO").is_not_null()
    )
    .sort([
        "DAY",
        "PTCAR_NO",
        "PLANNED_DATETIME_ARR"
    ])
)

intra_day.head(20)

DAY,TRAIN_ID,TRAIN_NO,ORD_NO,PTCAR_NO,PTCAR_NAME,PLANNED_DATETIME_ARR,REAL_DATETIME_ARR,DELAY_ARR_MIN
date,str,str,str,str,str,datetime[μs],datetime[μs],f64
2025-01-01,"""119539766""","""506""","""136""","""215""","""Bruxelles-Central""",2025-01-01 08:00:00,2025-01-01 08:00:00,0.0
2025-01-01,"""119550497""","""3107""","""130""","""215""","""Bruxelles-Central""",2025-01-01 08:03:00,2025-01-01 08:02:33,-0.433333
2025-01-01,"""119527481""","""3757""","""119""","""215""","""Bruxelles-Central""",2025-01-01 08:03:00,2025-01-01 08:04:19,1.316667
2025-01-01,"""119547285""","""1929""","""113""","""215""","""Bruxelles-Central""",2025-01-01 08:05:00,2025-01-01 08:04:35,-0.4
2025-01-01,"""119546275""","""1978""","""134""","""215""","""Bruxelles-Central""",2025-01-01 08:06:00,2025-01-01 08:06:08,0.133333
…,…,…,…,…,…,…,…,…
2025-01-01,"""119521691""","""2331""","""113""","""215""","""Bruxelles-Central""",2025-01-01 08:31:00,2025-01-01 08:31:34,0.566667
2025-01-01,"""119548369""","""3407""","""131""","""215""","""Bruxelles-Central""",2025-01-01 08:37:00,2025-01-01 08:36:44,-0.25
2025-01-01,"""119524817""","""3228""","""119""","""215""","""Bruxelles-Central""",2025-01-01 08:38:00,2025-01-01 08:37:43,-0.266667


In [ ]:
intra_day = (
    intra_day
    .with_columns([
        
        pl.col("TRAIN_ID")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("PREVIOUS_TRAIN_ID"),

        pl.col("TRAIN_NO")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("PREVIOUS_TRAIN_NO"),

        pl.col("PLANNED_DATETIME_ARR")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("PREVIOUS_TIME"),

        pl.col("DELAY_ARR_MIN")
        .shift(1)
        .over(["DAY", "PTCAR_NO"])
        .alias("PREVIOUS_DELAY_MIN")
    ])
)

In [ ]:
intra_day = (
    intra_day
    .with_columns(
        (
            (pl.col("PLANNED_DATETIME_ARR") - pl.col("PREVIOUS_TIME"))
            .dt.total_seconds()
            / 60
        ).alias("TIME_GAP_MIN")
    )
    .filter(
        pl.col("PREVIOUS_TRAIN_ID").is_not_null()
    )
)

In [ ]:
intra_day.select([
    "DAY",
    "PTCAR_NAME",
    "PREVIOUS_TRAIN_NO",
    "TRAIN_NO",
    "PREVIOUS_TIME",
    "PLANNED_DATETIME_ARR",
    "TIME_GAP_MIN",
    "PREVIOUS_DELAY_MIN",
    "DELAY_ARR_MIN"
]).head(50)

DAY,PTCAR_NAME,PREVIOUS_TRAIN_NO,TRAIN_NO,PREVIOUS_TIME,PLANNED_DATETIME_ARR,TIME_GAP_MIN,PREVIOUS_DELAY_MIN,DELAY_ARR_MIN
date,str,str,str,datetime[μs],datetime[μs],f64,f64,f64
2025-01-01,"""Bruxelles-Central""","""506""","""3107""",2025-01-01 08:00:00,2025-01-01 08:03:00,3.0,0.0,-0.433333
2025-01-01,"""Bruxelles-Central""","""3107""","""3757""",2025-01-01 08:03:00,2025-01-01 08:03:00,0.0,-0.433333,1.316667
2025-01-01,"""Bruxelles-Central""","""3757""","""1929""",2025-01-01 08:03:00,2025-01-01 08:05:00,2.0,1.316667,-0.4
2025-01-01,"""Bruxelles-Central""","""1929""","""1978""",2025-01-01 08:05:00,2025-01-01 08:06:00,1.0,-0.4,0.133333
2025-01-01,"""Bruxelles-Central""","""1978""","""6579""",2025-01-01 08:06:00,2025-01-01 08:08:00,2.0,0.133333,0.85
…,…,…,…,…,…,…,…,…
2025-01-01,"""Bruxelles-Congrès""","""2305""","""3228""",2025-01-01 08:30:00,2025-01-01 08:36:00,6.0,-0.45,0.166667
2025-01-01,"""Bruxelles-Congrès""","""3228""","""2830""",2025-01-01 08:36:00,2025-01-01 08:40:00,4.0,0.166667,0.4
2025-01-01,"""Bruxelles-Congrès""","""2830""","""3407""",2025-01-01 08:40:00,2025-01-01 08:40:00,0.0,0.4,-0.333333


In [ ]:
intra_day.select(
    "TIME_GAP_MIN"
).describe()

statistic,TIME_GAP_MIN
str,f64
"""count""",9919.0
"""null_count""",0.0
"""mean""",1.08267
"""std""",1.109258
"""min""",0.0
"""25%""",0.0
"""50%""",1.0
"""75%""",2.0
"""max""",8.0


In [ ]:
import plotly.express as px

gap_data = intra_day.select(
    "TIME_GAP_MIN"
).to_pandas()

fig = px.histogram(
    gap_data,
    x="TIME_GAP_MIN",
    nbins=50,
    title="Time Gap Between Consecutive Trains – 08:00–09:00"
)

fig.update_layout(
    xaxis_title="Time gap (minutes)",
    yaxis_title="Number of consecutive train pairs"
)

fig.show(renderer="browser")

In [ ]:
intra_day.select([
    pl.col("TIME_GAP_MIN").min().alias("MIN_GAP"),
    pl.col("TIME_GAP_MIN").median().alias("MEDIAN_GAP"),
    pl.col("TIME_GAP_MIN").mean().alias("MEAN_GAP"),
    pl.col("TIME_GAP_MIN").max().alias("MAX_GAP")
])

MIN_GAP,MEDIAN_GAP,MEAN_GAP,MAX_GAP
f64,f64,f64,f64
0.0,1.0,1.08267,8.0


In [ ]:
intra_day.select([
    (pl.col("TIME_GAP_MIN") <= 2).sum().alias("GAP_0_2_MIN"),
    ((pl.col("TIME_GAP_MIN") > 2) & (pl.col("TIME_GAP_MIN") <= 5))
        .sum().alias("GAP_2_5_MIN"),
    ((pl.col("TIME_GAP_MIN") > 5) & (pl.col("TIME_GAP_MIN") <= 10))
        .sum().alias("GAP_5_10_MIN"),
    (pl.col("TIME_GAP_MIN") > 10).sum().alias("GAP_OVER_10_MIN")
])

GAP_0_2_MIN,GAP_2_5_MIN,GAP_5_10_MIN,GAP_OVER_10_MIN
u32,u32,u32,u32
8927,942,50,0


In [ ]:
pair_sequences = (
    intra_day
    .group_by([
        "DAY",
        "PREVIOUS_TRAIN_ID",
        "PREVIOUS_TRAIN_NO",
        "TRAIN_ID",
        "TRAIN_NO"
    ])
    .agg([
        pl.len().alias("N_SHARED_POINTS"),

        pl.col("PTCAR_NO")
        .n_unique()
        .alias("N_PTCAR"),

        pl.col("TIME_GAP_MIN")
        .median()
        .alias("MEDIAN_GAP_MIN"),

        pl.col("PREVIOUS_DELAY_MIN")
        .first()
        .alias("LEADER_DELAY_START"),

        pl.col("PREVIOUS_DELAY_MIN")
        .last()
        .alias("LEADER_DELAY_END"),

        pl.col("DELAY_ARR_MIN")
        .first()
        .alias("FOLLOWER_DELAY_START"),

        pl.col("DELAY_ARR_MIN")
        .last()
        .alias("FOLLOWER_DELAY_END")
    ])

    .sort(
        ["N_PTCAR", "N_SHARED_POINTS"],
        descending=True
    )
)

pair_sequences.head(30)

DAY,PREVIOUS_TRAIN_ID,PREVIOUS_TRAIN_NO,TRAIN_ID,TRAIN_NO,N_SHARED_POINTS,N_PTCAR,MEDIAN_GAP_MIN,LEADER_DELAY_START,LEADER_DELAY_END,FOLLOWER_DELAY_START,FOLLOWER_DELAY_END
date,str,str,str,str,u32,u32,f64,f64,f64,f64,f64
2025-01-13,"""119918786""","""2207""","""119909375""","""2257""",6,6,3.0,2.233333,1.716667,4.783333,0.85
2025-01-04,"""119629422""","""6557""","""119616332""","""1958""",6,6,1.0,0.583333,0.283333,0.766667,0.05
2025-01-26,"""120379838""","""1529""","""120384616""","""3429""",6,6,3.0,0.15,0.55,-0.45,0.233333
2025-01-25,"""120352323""","""3429""","""120347100""","""2128""",6,6,1.0,0.633333,1.083333,0.066667,0.75
2025-01-05,"""119644665""","""1929""","""119644613""","""1978""",5,5,1.0,-0.166667,0.533333,0.6,0.433333
…,…,…,…,…,…,…,…,…,…,…,…
2025-01-08,"""119759085""","""7076""","""119739523""","""2128""",5,5,0.0,1.883333,1.55,-0.366667,-0.366667
2025-01-13,"""119910825""","""2279""","""119915859""","""1529""",5,5,8.0,18.55,19.5,5.6,7.133333
2025-01-21,"""120203901""","""3629""","""120203828""","""2830""",5,5,0.0,2.666667,3.05,2.366667,3.05


In [ ]:
pair_sequences.select([
    "DAY",
    "PREVIOUS_TRAIN_NO",
    "TRAIN_NO",
    "N_SHARED_POINTS",
    "N_PTCAR",
    "MEDIAN_GAP_MIN"
]).head(30)

DAY,PREVIOUS_TRAIN_NO,TRAIN_NO,N_SHARED_POINTS,N_PTCAR,MEDIAN_GAP_MIN
date,str,str,u32,u32,f64
2025-01-13,"""2207""","""2257""",6,6,3.0
2025-01-04,"""6557""","""1958""",6,6,1.0
2025-01-26,"""1529""","""3429""",6,6,3.0
2025-01-25,"""3429""","""2128""",6,6,1.0
2025-01-05,"""1929""","""1978""",5,5,1.0
…,…,…,…,…,…
2025-01-08,"""7076""","""2128""",5,5,0.0
2025-01-13,"""2279""","""1529""",5,5,8.0
2025-01-21,"""3629""","""2830""",5,5,0.0


In [ ]:
pair_sequences.select([
    pl.col("N_SHARED_POINTS").min().alias("MIN_POINTS"),
    pl.col("N_SHARED_POINTS").median().alias("MEDIAN_POINTS"),
    pl.col("N_SHARED_POINTS").mean().alias("MEAN_POINTS"),
    pl.col("N_SHARED_POINTS").max().alias("MAX_POINTS")
])

MIN_POINTS,MEDIAN_POINTS,MEAN_POINTS,MAX_POINTS
u32,f64,f64,u32
1,1.0,1.264533,6


In [ ]:
pair_sequences.filter(
    pl.col("N_SHARED_POINTS") >= 10
).height

0

In [ ]:
domino_events = (
    intra_day

    # Order observations along the journey
    .sort([
        "DAY",
        "PREVIOUS_TRAIN_ID",
        "TRAIN_ID",
        "PLANNED_DATETIME_ARR"
    ])

    .group_by([
        "DAY",
        "PREVIOUS_TRAIN_ID",
        "PREVIOUS_TRAIN_NO",
        "TRAIN_ID",
        "TRAIN_NO"
    ])

    .agg([
        # Number of operational points where they follow each other
        pl.len().alias("N_SHARED_POINTS"),

        # Typical distance in time between the two trains
        pl.col("TIME_GAP_MIN")
        .median()
        .alias("MEDIAN_TIME_GAP_MIN"),

        # Leader delay
        pl.col("PREVIOUS_DELAY_MIN")
        .first()
        .alias("LEADER_DELAY_START"),

        pl.col("PREVIOUS_DELAY_MIN")
        .last()
        .alias("LEADER_DELAY_END"),

        # Follower delay
        pl.col("DELAY_ARR_MIN")
        .first()
        .alias("FOLLOWER_DELAY_START"),

        pl.col("DELAY_ARR_MIN")
        .last()
        .alias("FOLLOWER_DELAY_END")
    ])

    .rename({
        "PREVIOUS_TRAIN_ID": "LEADER_TRAIN_ID",
        "PREVIOUS_TRAIN_NO": "LEADER_TRAIN_NO",
        "TRAIN_ID": "FOLLOWER_TRAIN_ID",
        "TRAIN_NO": "FOLLOWER_TRAIN_NO"
    })

    # Calculate how much delay was gained/lost
    .with_columns([
        (
            pl.col("LEADER_DELAY_END")
            - pl.col("LEADER_DELAY_START")
        ).alias("LEADER_DELAY_CHANGE"),

        (
            pl.col("FOLLOWER_DELAY_END")
            - pl.col("FOLLOWER_DELAY_START")
        ).alias("FOLLOWER_DELAY_CHANGE")
    ])

    .sort("DAY")
)

In [ ]:
domino_events.select([
    "DAY",
    "LEADER_TRAIN_NO",
    "FOLLOWER_TRAIN_NO",
    "N_SHARED_POINTS",
    "MEDIAN_TIME_GAP_MIN",
    "LEADER_DELAY_START",
    "LEADER_DELAY_END",
    "LEADER_DELAY_CHANGE",
    "FOLLOWER_DELAY_START",
    "FOLLOWER_DELAY_END",
    "FOLLOWER_DELAY_CHANGE"
]).head(50)

DAY,LEADER_TRAIN_NO,FOLLOWER_TRAIN_NO,N_SHARED_POINTS,MEDIAN_TIME_GAP_MIN,LEADER_DELAY_START,LEADER_DELAY_END,LEADER_DELAY_CHANGE,FOLLOWER_DELAY_START,FOLLOWER_DELAY_END,FOLLOWER_DELAY_CHANGE
date,str,str,u32,f64,f64,f64,f64,f64,f64,f64
2025-01-01,"""3407""","""1507""",4,3.0,-0.5,-0.333333,0.166667,-1.45,0.116667,1.566667
2025-01-01,"""3779""","""3129""",1,1.0,-0.35,-0.35,0.0,-0.583333,-0.583333,0.0
2025-01-01,"""3707""","""3329""",1,3.0,-0.85,-0.85,0.0,0.5,0.5,0.0
2025-01-01,"""1958""","""2078""",1,1.0,0.366667,0.366667,0.0,3.133333,3.133333,0.0
2025-01-01,"""1929""","""6579""",1,2.0,-0.583333,-0.583333,0.0,0.816667,0.816667,0.0
…,…,…,…,…,…,…,…,…,…,…
2025-01-01,"""1978""","""3107""",1,1.0,-0.333333,-0.333333,0.0,-0.083333,-0.083333,0.0
2025-01-01,"""3429""","""2806""",1,0.0,0.766667,0.766667,0.0,0.433333,0.433333,0.0
2025-01-01,"""6579""","""3707""",2,3.0,0.816667,0.85,0.033333,-0.033333,-0.616667,-0.583333


In [ ]:
domino_summary = domino_events.select([
    pl.len().alias("N_EVENTS"),
    pl.col("DAY").n_unique().alias("N_DAYS"),

    pl.col("LEADER_TRAIN_ID").n_unique().alias("N_LEADERS"),
    pl.col("FOLLOWER_TRAIN_ID").n_unique().alias("N_FOLLOWERS"),

    pl.col("N_SHARED_POINTS").mean().alias("MEAN_SHARED_POINTS"),

    pl.col("MEDIAN_TIME_GAP_MIN").median().alias("MEDIAN_GAP_MIN"),

    pl.col("LEADER_DELAY_CHANGE").mean().alias("MEAN_LEADER_DELAY_CHANGE"),

    pl.col("FOLLOWER_DELAY_CHANGE").mean().alias("MEAN_FOLLOWER_DELAY_CHANGE")
])

domino_summary

N_EVENTS,N_DAYS,N_LEADERS,N_FOLLOWERS,MEAN_SHARED_POINTS,MEDIAN_GAP_MIN,MEAN_LEADER_DELAY_CHANGE,MEAN_FOLLOWER_DELAY_CHANGE
u32,u32,u32,u32,f64,f64,f64,f64
7844,31,2008,1984,1.264533,1.0,0.110288,0.133484


In [ ]:
domino_events.select([
    pl.col("FOLLOWER_DELAY_CHANGE").min().alias("MIN"),
    pl.col("FOLLOWER_DELAY_CHANGE").quantile(0.25).alias("Q1"),
    pl.col("FOLLOWER_DELAY_CHANGE").median().alias("MEDIAN"),
    pl.col("FOLLOWER_DELAY_CHANGE").mean().alias("MEAN"),
    pl.col("FOLLOWER_DELAY_CHANGE").quantile(0.75).alias("Q3"),
    pl.col("FOLLOWER_DELAY_CHANGE").max().alias("MAX")
])

MIN,Q1,MEDIAN,MEAN,Q3,MAX
f64,f64,f64,f64,f64,f64
-6.183333,0.0,0.0,0.133484,0.0,21.4


In [ ]:
import plotly.express as px

domino_pd = domino_events.to_pandas()

fig = px.scatter(
    domino_pd,
    x="LEADER_DELAY_START",
    y="FOLLOWER_DELAY_CHANGE",
    hover_data=[
        "DAY",
        "LEADER_TRAIN_NO",
        "FOLLOWER_TRAIN_NO",
        "MEDIAN_TIME_GAP_MIN",
        "N_SHARED_POINTS"
    ],
    title="Leader Delay vs Follower Delay Change"
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.update_layout(
    xaxis_title="Leader Delay at Start (minutes)",
    yaxis_title="Follower Delay Change (minutes)"
)

fig.show(renderer="browser")

In [ ]:
fig = px.scatter(
    domino_pd,
    x="MEDIAN_TIME_GAP_MIN",
    y="FOLLOWER_DELAY_CHANGE",
    color="LEADER_DELAY_START",
    hover_data=[
        "DAY",
        "LEADER_TRAIN_NO",
        "FOLLOWER_TRAIN_NO",
        "N_SHARED_POINTS"
    ],
    title="Follower Delay Change vs Time Gap"
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.update_layout(
    xaxis_title="Leader–Follower Time Gap (minutes)",
    yaxis_title="Follower Delay Change (minutes)"
)

fig.show(renderer="browser")

In [ ]:
domino_events = domino_events.with_columns(
    pl.when(pl.col("MEDIAN_TIME_GAP_MIN") <= 2)
      .then(pl.lit("0-2 min"))
      .when(pl.col("MEDIAN_TIME_GAP_MIN") <= 5)
      .then(pl.lit("2-5 min"))
      .when(pl.col("MEDIAN_TIME_GAP_MIN") <= 10)
      .then(pl.lit("5-10 min"))
      .otherwise(pl.lit(">10 min"))
      .alias("GAP_GROUP")
)

In [ ]:
gap_analysis = (
    domino_events
    .group_by("GAP_GROUP")
    .agg([
        pl.len().alias("N_EVENTS"),

        pl.col("FOLLOWER_DELAY_CHANGE")
          .mean()
          .alias("MEAN_FOLLOWER_DELAY_CHANGE"),

        pl.col("FOLLOWER_DELAY_CHANGE")
          .median()
          .alias("MEDIAN_FOLLOWER_DELAY_CHANGE"),

        (pl.col("FOLLOWER_DELAY_CHANGE") > 0)
          .mean()
          .alias("PROP_FOLLOWER_GAINED_DELAY"),

        pl.col("LEADER_DELAY_START")
          .mean()
          .alias("MEAN_LEADER_DELAY")
    ])
    .sort("GAP_GROUP")
)

gap_analysis

GAP_GROUP,N_EVENTS,MEAN_FOLLOWER_DELAY_CHANGE,MEDIAN_FOLLOWER_DELAY_CHANGE,PROP_FOLLOWER_GAINED_DELAY,MEAN_LEADER_DELAY
str,u32,f64,f64,f64,f64
"""0-2 min""",7075,0.141984,0.0,0.118304,4.781091
"""2-5 min""",731,0.055267,0.0,0.101231,3.980255
"""5-10 min""",38,0.055702,0.0,0.052632,2.353509


In [ ]:
domino_events = domino_events.with_columns(
    (pl.col("LEADER_DELAY_START") >= 6)
    .alias("LEADER_DELAYED")
)

In [ ]:
domino_comparison = (
    domino_events
    .group_by([
        "GAP_GROUP",
        "LEADER_DELAYED"
    ])
    .agg([
        pl.len().alias("N_EVENTS"),

        pl.col("FOLLOWER_DELAY_CHANGE")
          .mean()
          .alias("MEAN_FOLLOWER_DELAY_CHANGE"),

        (pl.col("FOLLOWER_DELAY_CHANGE") > 0)
          .mean()
          .alias("PROP_FOLLOWER_GAINED_DELAY")
    ])
    .sort([
        "GAP_GROUP",
        "LEADER_DELAYED"
    ])
)

domino_comparison

GAP_GROUP,LEADER_DELAYED,N_EVENTS,MEAN_FOLLOWER_DELAY_CHANGE,PROP_FOLLOWER_GAINED_DELAY
str,bool,u32,f64,f64
"""0-2 min""",false,5373,0.140161,0.12265
"""0-2 min""",true,1702,0.147738,0.104583
"""2-5 min""",false,581,0.052869,0.101549
"""2-5 min""",true,150,0.064556,0.1
"""5-10 min""",false,34,0.010784,0.029412
"""5-10 min""",true,4,0.4375,0.25


In [ ]:
domino_events.select([
    pl.corr(
        "LEADER_DELAY_CHANGE",
        "FOLLOWER_DELAY_CHANGE"
    ).alias("CORR_DELAY_CHANGES")
])

CORR_DELAY_CHANGES
f64
0.300546


In [ ]:
domino_events.filter(
    pl.col("MEDIAN_TIME_GAP_MIN") <= 2
).select([
    pl.len().alias("N_EVENTS"),

    pl.corr(
        "LEADER_DELAY_CHANGE",
        "FOLLOWER_DELAY_CHANGE"
    ).alias("CORR_DELAY_CHANGES")
])

N_EVENTS,CORR_DELAY_CHANGES
u32,f64
7075,0.303882


Pour les trains Leader–Follower très proches, avec un écart temporel inférieur ou égal à 2 minutes, nous disposons de 7 075 événements. La corrélation entre la variation du retard du Leader et celle du Follower est de 0,304, ce qui indique une association positive modérée. Autrement dit, lorsque le Leader accumule du retard pendant la traversée de la jonction Nord–Midi, le Follower tend également à accumuler du retard. Ce résultat constitue un signal compatible avec l’existence potentielle d’un effet domino. Cependant, il ne permet pas à lui seul de conclure à une relation causale, car des facteurs communs, tels que les incidents, la congestion ou d’autres contraintes opérationnelles, peuvent influencer simultanément les deux trains.

# Granger a été testé, mais il n'a pas permis de démontrer une causalité